# GEPA Medical Condition Classification

Use the `sv_medical_200.csv` dataset, take `patient_notes` as input, and predict the gold medical condition in `label`.
This notebook follows the same staged structure as `gepa_3_tldr.ipynb`, but uses exact-match classification accuracy as the primary optimization target.

In [1]:
# ============================================================
# 0) IMPORTS & CONFIG
# ============================================================
import csv
import os
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import dspy


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


#REPO_ROOT = find_repo_root()
#DATA_PATH = REPO_ROOT / "data" / "sv_medical_200.csv"
DATA_PATH ="/Users/sangeetha/Supportvector2026/agents/lab/dspy_basics/src/hackathon/hackathon_final_200_april6.csv"
# Base model for reasoning
MODEL_NAME = os.getenv("DSPY_MODEL", "openai/openai/gpt-oss-20b")
API_BASE = os.getenv("DSPY_API_BASE", "http://10.0.10.51:8123/v1")
API_KEY = os.getenv("DSPY_API_KEY", "sv-openai-api-key")


lm=dspy.LM(
    # 'openai/gpt-4o-mini'
    "openai/openai/gpt-oss-20b",
    api_base="http://10.0.10.51:8124/v1",
    api_key="sv-openai-api-key",
    
)

dspy.configure(lm=lm)

# Stronger / more exploratory LM for reflection
reflection_lm = dspy.LM(
    # 'openai/gpt-4o-mini'
    "openai/openai/gpt-oss-20b",
    api_base="http://10.0.10.51:8124/v1",
    api_key="sv-openai-api-key",
)

#print(f"Repo root: {REPO_ROOT}")
print(f"Dataset path: {DATA_PATH}")
print(f"Model: {MODEL_NAME}")

Dataset path: /Users/sangeetha/Supportvector2026/agents/lab/dspy_basics/src/hackathon/hackathon_final_200_april6.csv
Model: openai/openai/gpt-oss-20b


In [2]:
# ============================================================
# 1) LOAD DATASET + STRATIFIED SPLITS
# ============================================================
def load_medical_rows(path: Path):
    with path.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))

    cleaned = []
    for row in rows:
        patient_notes = row["patient_notes"].strip()
        label = row["label"].strip()
        reasoning = row.get("reasoning", "").strip()
        if patient_notes and label:
            cleaned.append(
                {
                    "patient_notes": patient_notes,
                    "label": label,
                    "reasoning": reasoning,
                }
            )
    return cleaned


def stratified_split(rows, train_ratio=0.70, val_ratio=0.15, seed=7):
    rng = random.Random(seed)
    buckets = defaultdict(list)
    for row in rows:
        buckets[row["label"]].append(row)

    train_rows, val_rows, test_rows = [], [], []
    for label, items in sorted(buckets.items()):
        items = items[:]
        rng.shuffle(items)

        n_total = len(items)
        n_train = max(1, int(round(n_total * train_ratio)))
        n_val = max(1, int(round(n_total * val_ratio)))

        if n_train + n_val >= n_total:
            n_val = max(1, n_total - n_train - 1)
        n_test = n_total - n_train - n_val
        if n_test <= 0:
            n_test = 1
            if n_train > n_val:
                n_train -= 1
            else:
                n_val -= 1

        train_rows.extend(items[:n_train])
        val_rows.extend(items[n_train:n_train + n_val])
        test_rows.extend(items[n_train + n_val:])

    rng.shuffle(train_rows)
    rng.shuffle(val_rows)
    rng.shuffle(test_rows)
    return train_rows, val_rows, test_rows


def to_examples(rows):
    return [
        dspy.Example(
            {
                "patient_notes": row["patient_notes"],
                "gold_label": row["label"],
                "reasoning": row["reasoning"],
            }
        ).with_inputs("patient_notes")
        for row in rows
    ]

DATA_PATH = Path("/Users/sangeetha/Supportvector2026/agents/lab/dspy_basics/src/hackathon/hackathon_final_200_april6.csv")
rows = load_medical_rows(DATA_PATH)
train_rows, val_rows, test_rows = stratified_split(rows)

train_set = to_examples(train_rows)
val_set = to_examples(val_rows)
test_set = to_examples(test_rows)

label_counts = Counter(row["label"] for row in rows)
label_space = sorted(label_counts)

print(f"Total rows: {len(rows)}")
print(f"Train / Val / Test: {len(train_set)} / {len(val_set)} / {len(test_set)}")
print("\nLabel distribution:")
for label, count in sorted(label_counts.items()):
    print(f"- {label}: {count}")

print("\nAllowed labels:")
print(label_space)

Total rows: 205
Train / Val / Test: 144 / 30 / 31

Label distribution:
- Arthritis: 34
- COPD/Respiratory: 34
- Cardiovascular: 34
- Diabetes: 34
- Eye Condition: 34
- Neurological: 35

Allowed labels:
['Arthritis', 'COPD/Respiratory', 'Cardiovascular', 'Diabetes', 'Eye Condition', 'Neurological']


In [3]:
# ============================================================
# 2) SIGNATURE + MODULE
# ============================================================
ALLOWED_LABELS_TEXT = ", ".join(label_space)


class PredictMedicalCondition(dspy.Signature):
    """
    Read the patient notes and predict the single best medical condition label.
    """

    patient_notes: str = dspy.InputField(desc="Patient history, symptoms, and clinical notes")
    medical_condition: str = dspy.OutputField(desc="One medical condition")


class MedicalConditionModule(dspy.Module):
    def __init__(self):
        self.generator = dspy.ChainOfThought(PredictMedicalCondition)

    def forward(self, patient_notes: str):
        out = self.generator(patient_notes=patient_notes)
        return dspy.Prediction(medical_condition=out.medical_condition)


program = MedicalConditionModule()
print(ALLOWED_LABELS_TEXT)

Arthritis, COPD/Respiratory, Cardiovascular, Diabetes, Eye Condition, Neurological


In [4]:
# ============================================================
# 3) HELPERS (label normalization, reporting, evaluation)
# ============================================================
CANONICAL_LABELS = {re.sub(r'[^a-z0-9]+', '', label.lower()): label for label in label_space}


def normalize_label(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", (text or "").strip().lower())


def canonicalize_label(text: str) -> str:
    return CANONICAL_LABELS.get(normalize_label(text), (text or "").strip())


def is_valid_label(text: str) -> bool:
    return normalize_label(text) in CANONICAL_LABELS


def exact_match_score(gold: str, pred: str) -> float:
    return 1.0 if normalize_label(gold) == normalize_label(pred) else 0.0


def dataset_accuracy(module, dataset):
    correct = 0
    total = len(dataset)
    rows = []

    for example in dataset:
        pred = module(patient_notes=example["patient_notes"])
        pred_label = canonicalize_label(pred.medical_condition)
        gold_label = example["gold_label"]
        score = exact_match_score(gold_label, pred_label)
        correct += score
        rows.append(
            {
                "gold": gold_label,
                "pred": pred_label,
                "raw_pred": pred.medical_condition,
                "correct": bool(score),
            }
        )

    accuracy = correct / total if total else 0.0
    return accuracy, rows


def print_sample_predictions(results, n=5):
    for i, row in enumerate(results[:n], start=1):
        print(f"Example {i}: gold={row['gold']} | pred={row['pred']} | correct={row['correct']}")
        if row["raw_pred"] != row["pred"]:
            print(f"  raw_pred={row['raw_pred']}")

In [5]:
# ============================================================
# 4) FEEDBACK FUNCTIONS — OPTIMIZED FOR CLASSIFICATION ACCURACY
# ============================================================
def feedback_valid_label(pred_label: str):
    valid = is_valid_label(pred_label)
    score = 1.0 if valid else 0.0

    if valid:
        fb = f"You predicted a valid label: {canonicalize_label(pred_label)}."
    else:
        fb = (
            "Your output is not one of the allowed labels. "
            f"Choose exactly one from: {ALLOWED_LABELS_TEXT}."
        )
    return fb, score


def feedback_accuracy(example, pred_label: str):
    gold_label = example["gold_label"]
    pred_canonical = canonicalize_label(pred_label)
    score = exact_match_score(gold_label, pred_canonical)

    if score == 1.0:
        fb = f"Correct. The predicted label matches the gold label: {gold_label}."
    else:
        fb = (
            f"Incorrect. You predicted '{pred_canonical}' but the gold label is '{gold_label}'. "
            "Pay closer attention to symptom patterns, organ system clues, and disease-specific terminology in the notes."
        )
    return fb, score


In [6]:
# ============================================================
# 5) METRIC (SCALAR ONLY)
# ============================================================
def metric(example, pred, trace=None, pred_name=None, pred_trace=None):
    _, s_valid = feedback_valid_label(pred.medical_condition)
    _, s_acc = feedback_accuracy(example, pred.medical_condition)

    total = (0.10 * s_valid) + (0.90 * s_acc)
    return total


In [7]:
# ============================================================
# BASELINE EVALUATION
# ============================================================
evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=8,
    display_table=True,
    display_progress=True,
)

print("\n== Baseline Evaluation ==")
evaluate(program)

baseline_accuracy, baseline_rows = dataset_accuracy(program, test_set)
print(f"\nBaseline exact-match accuracy: {baseline_accuracy:.2%}")
print_sample_predictions(baseline_rows)


== Baseline Evaluation ==
Average Metric: 0.00 / 31 (0.0%): 100%|██████████| 31/31 [00:29<00:00,  1.06it/s]

2026/04/06 08:39:09 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 31 (0.0%)


,patient_notes,gold_label,reasoning,medical_condition,metric
0,67 y/o M presents with new onset profound fatigue and lightheadedn...,Cardiovascular,Profound fatigue and lightheadedness in an elderly patient with T2...,Acute decompensated systolic heart failure,✔️ [0.000]
1,"68 y/o M c/o generalized fatigue and weakness x weeks, some dizzin...",Cardiovascular,Initial symptoms of fatigue and dizziness are non-specific and cou...,Congestive heart failure (systolic HF),✔️ [0.000]
2,50 y.o. F c/o progressive bilateral lower extremity weakness and n...,Diabetes,"The progressive bilateral lower extremity weakness and numbness, a...",Guillain-Barré syndrome,✔️ [0.000]
3,60 y.o. M presents c/o several episodes of transient vision loss i...,Arthritis,The transient monocular vision loss (amaurosis fugax) is a critica...,Rheumatoid arthritis,✔️ [0.000]
4,70 y/o F c/o increasing generalized weakness and fatigue for 3 mon...,Neurological,While generalized weakness and fatigue are common in polymyalgia r...,Polymyositis,✔️ [0.000]
5,"28 y.o. M presents c/o acute onset dyspnea and wheezing, unrespons...",Arthritis,The acute dyspnea and wheezing strongly point to an asthma exacerb...,Septic arthritis,✔️ [0.000]
6,"53 y.o. F here for chronic heartburn and regurgitation, worse at n...",COPD/Respiratory,Heartburn and regurgitation point to a GI issue (red herring). Eso...,Gastroesophageal Reflux Disease (GERD),✔️ [0.000]
7,"68 y/o M presents with gross hematuria, flank pain, and dysuria fo...",Neurological,"The initial symptoms are classic for a urological issue, likely re...",Benign prostatic hyperplasia (BPH) with acute urinary retention,✔️ [0.000]
8,67 y/o M brought in for increasing difficulty with balance and fre...,Neurological,"While balance issues and falls can be multi-factorial, the specifi...",Cerebellar ataxia,✔️ [0.000]
9,33 y.o. F c/o right ear pain and tinnitus x 2 days. Hx of TMJ dysf...,COPD/Respiratory,Ear pain and tinnitus point to an ENT issue (red herring). TMJ dys...,Temporomandibular joint disorder,✔️ [0.000]



Baseline exact-match accuracy: 0.00%
Example 1: gold=Cardiovascular | pred=Acute decompensated systolic heart failure | correct=False
Example 2: gold=Cardiovascular | pred=Congestive heart failure (systolic HF) | correct=False
Example 3: gold=Diabetes | pred=Guillain-Barré syndrome | correct=False
Example 4: gold=Arthritis | pred=Rheumatoid arthritis | correct=False
Example 5: gold=Neurological | pred=Polymyositis | correct=False


In [8]:
# ============================================================
# 6) METRIC WITH FEEDBACK (GEPA-READY)
# ============================================================
def metric_with_feedback(example, pred, trace=None, pred_name=None, pred_trace=None):
    fb_valid, s_valid = feedback_valid_label(pred.medical_condition)
    fb_acc, s_acc = feedback_accuracy(example, pred.medical_condition)

    total = (0.10 * s_valid) + (0.90 * s_acc)

    if pred_name is None:
        return total

    if pred_name == "generator.predict":
        feedback = (
            f"Label Validity Feedback:\n{fb_valid}\n\n"
            f"Accuracy Feedback:\n{fb_acc}\n\n"
            f"Allowed Labels:\n{ALLOWED_LABELS_TEXT}\n\n"
            f"Overall Score: {total:.3f}\n"
            "Think step-by-step about the organ system, hallmark symptoms, chronic disease context, and the single best label to return."
        )
    else:
        feedback = "No specific predictor matched. Review the classification reasoning carefully."

    return dspy.Prediction(score=total, feedback=feedback)

In [9]:
# ============================================================
# 7) GEPA OPTIMIZER
# ============================================================
from dspy import GEPA

optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light",
    num_threads=8,
    track_stats=True,
    track_best_outputs=True,
    use_merge=False,
    reflection_lm=reflection_lm,
)

optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    valset=val_set,
)

2026/04/06 08:41:54 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 500 metric calls of the program. This amounts to 2.87 full evals on the train+val set.
2026/04/06 08:41:54 INFO dspy.teleprompt.gepa.gepa: Using 30 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|          | 0/500 [00:00<?, ?rollouts/s]2026/04/06 08:42:22 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 30 (0.0%)
2026/04/06 08:42:22 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.0 over 30 / 30 examples
GEPA Optimization:   6%|▌         | 30/500 [00:27<07:17,  1.07rollouts/s]2026/04/06 08:42:22 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.0


Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:07<00:00,  2.46s/it]

2026/04/06 08:42:29 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 3 (0.0%)


2026/04/06 08:42:36 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for generator.predict: ## Task: Single‑Label Disease Classification from Note Extractions

You will receive a JSON object that contains a field `patient_notes` with a free‑text clinical vignette.  
Your job is to read the notes, identify the most relevant organ system and clinical presentation, and then output **exactly one** of the following six labels:

- **Arthritis**  
- **COPD/Respiratory**  
- **Cardiovascular**  
- **Diabetes**  
- **Eye Condition**  
- **Neurological**

### Output Format

Your response must consist of **two** parts in the exact order shown:

```
reasoning: <Your short, step‑by‑step explanation of how you mapped the vignette to a label. Keep it concise (1‑3 sentences).>

medical_condition: <One of the six allowed labels>
```

Do **not** include any other text, formatting, or labels. The `reasoning` field may only contain plain text, no markdown formatting. The `medical_condition` 

Average Metric: 2.10 / 3 (70.0%): 100%|██████████| 3/3 [00:04<00:00,  1.35s/it] 

...

Average Metric: 0.10 / 1 (10.0%):  33%|███▎      | 1/3 [00:02<00:05,  2.68s/it]

2026/04/06 08:56:16 ERROR dspy.utils.parallelizer: Error for Example({'patient_notes': 'Pt, 63 y.o. M, presents with sudden onset monocular perception loss. H/o polycystic kidney disease, HTN, DM2. Denies pain. BP 190/100. Labs: Cr 2.5. R/o hypertensive crisis. Pt recently argued with his neighbor. Sudden, painless monocular perception loss noted, and papilla appears markedly edematous with splinter hemorrhages.', 'gold_label': 'Eye Condition', 'reasoning': "The sudden monocular perception loss, combined with severe hypertension in a patient with CKD and DM2, could be attributed to a hypertensive crisis affecting end organs. The neighbor argument is noise. However, the final sentence describing 'sudden, painless monocular perception loss' and a 'markedly edematous papilla with splinter hemorrhages' is a classic presentation of an 'Eye Condition' (specifically, acute interruption of blood supply to the nerve tissue in the posterior pole), overriding the hypertensive emergency."}) (input

Average Metric: 1.10 / 2 (55.0%): 100%|██████████| 3/3 [00:11<00:00,  3.86s/it]

2026/04/06 08:56:17 INFO dspy.evaluate.evaluate: Average Metric: 1.1 / 3 (36.7%)
2026/04/06 08:56:17 WARNING dspy.teleprompt.bootstrap_trace: Failed to unpack prediction and trace. This is likely due to the LLM response not following dspy formatting.


2026/04/06 08:56:27 INFO dspy.teleprompt.gepa.gepa: Iteration 29: Proposed new text for generator.predict: ## Single‑Label Organ‑System Classification from Clinical Vignettes

### 1. Purpose
From a free‑text clinical vignette determine **one** organ‑system category that best explains the patient’s primary problem.  You must provide that category and a brief, evidence‑based justification.

### 2. Input Format
The user will supply **exactly one** JSON object with a single key:

```json
{
  "patient_notes": "<clinical vignette>"
}
```

The string may include chief complaint, history of present illness, review of systems, past medical history, medications, physical exam, laboratory data, imaging, and any other relevant detail.  Do **not** alter, truncate, or otherwise process more than the provided text.

### 3. Allowed Output Labels  
You must choose **exactly one** of the following case‑sensitive labels:

```
Arthritis
COPD/Respiratory
Cardiovascular
Diabetes
Eye Condition
Neurological
`

In [10]:
# ============================================================
# 8) SEE OPTIMIZED PROMPTS
# ============================================================
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")

Predictor: generator.predict
Prompt:
You are provided with a block of patient notes that describe symptoms, history, exam findings, and test results.  
Your job is to assign one of six possible medical categories **without naming any specific disease**.  

**Output format (must match exactly):**  
```
### reasoning
<1–3 concise sentences, each expressing a single idea>
 
### medical_condition
<one of: Arthritis, COPD/Respiratory, Cardiovascular, Diabetes, Eye Condition, Neurological>
*********************************


In [11]:
# ============================================================
# 9) FINAL POST-GEPA EVALUATION
# ============================================================
print("\n== GEPA-Optimized Evaluation ==")
evaluate(optimized_program)

optimized_accuracy, optimized_rows = dataset_accuracy(optimized_program, test_set)
print(f"\nGEPA exact-match accuracy: {optimized_accuracy:.2%}")
print(f"Accuracy lift: {(optimized_accuracy - baseline_accuracy):+.2%}")
print_sample_predictions(optimized_rows)


== GEPA-Optimized Evaluation ==
Average Metric: 24.40 / 30 (81.3%): 100%|██████████| 30/30 [00:31<00:00,  1.05s/it]

2026/04/01 21:21:27 INFO dspy.evaluate.evaluate: Average Metric: 24.4 / 30 (81.3%)


,patient_notes,gold_label,reasoning,medical_condition,metric
0,12-year-old female whose parents noticed progressive pallor of her...,Eye Condition,The presence of pallor of the optic discs and severely reduced vis...,Eye Condition,✔️ [1.000]
1,"A 78-year-old female, recently discharged from the hospital after ...",COPD/Respiratory,The patient's recent stroke and dysphagia are significant risk fac...,COPD/Respiratory,✔️ [1.000]
2,19-year-old female presents after an episode of syncope while swim...,Cardiovascular,The occurrence of syncope during exertion (swimming) in a young in...,Cardiovascular,✔️ [1.000]
3,A 75-year-old male with a history of chronic kidney disease and di...,Arthritis,While the polyarticular involvement might initially suggest rheuma...,Arthritis,✔️ [1.000]
4,"81-year-old female admitted after a fall at home, sustaining a hip...",Cardiovascular,"The patient's recurrent lightheadedness, dizziness, and syncope up...",Cardiovascular,✔️ [1.000]
5,"An 85-year-old male, residing in a skilled nursing facility, prese...",COPD/Respiratory,"The patient's advanced age, institutionalization, acute onset of f...",COPD/Respiratory,✔️ [1.000]
6,A 58-year-old obese male (BMI 38 kg/m²) presents with chronic dayt...,COPD/Respiratory,While ankle edema and dyspnea might suggest primary cardiac or ren...,### medical_condition\nCardiovascular,✔️ [0.000]
7,92-year-old female found unresponsive at home. Paramedics report p...,Cardiovascular,"The presentation of sudden collapse, profound hypotension, a pulsa...",Cardiovascular,✔️ [1.000]
8,"A 72-year-old male, with a long-standing history of tobacco abuse ...",COPD/Respiratory,"The patient's advanced age, extensive smoking history, chronic pro...",COPD/Respiratory,✔️ [1.000]
9,A 9-month-old male infant presents with a 3-day history of rhinorr...,COPD/Respiratory,"The patient's age, viral prodrome, and classic signs of respirator...",COPD/Respiratory,✔️ [1.000]



GEPA exact-match accuracy: 80.00%
Accuracy lift: +80.00%
Example 1: gold=Eye Condition | pred=Eye Condition | correct=True
Example 2: gold=COPD/Respiratory | pred=COPD/Respiratory | correct=True
Example 3: gold=Cardiovascular | pred=Cardiovascular | correct=True
Example 4: gold=Arthritis | pred=Arthritis | correct=True
Example 5: gold=Cardiovascular | pred=Cardiovascular | correct=True


## Performance Improvement Summary

After running the notebook, fill in the baseline accuracy, optimized accuracy, and absolute lift on the held-out test split.

| Metric | Value |
|--------|-------|
| Baseline exact-match accuracy | `TBD` |
| GEPA exact-match accuracy | `TBD` |
| Accuracy lift | `TBD` |